# Avito DS Bootcamp: поиск и ранжирование объявлений

Цель — для каждого запроса вернуть 50 объявлений и максимизировать Recall@50. Решение v4 сначала собирает кандидатов несколькими способами, затем ранжирует их CatBoost YetiRank.

Notebook — исполняемый walkthrough того же кода, который запускается командой `python scripts/reproduce_v4.py --data-dir data`.

## 1. Что дано и как запустить notebook

В train есть запрос и выбранное объявление. В benchmark запросы и корпус объявлений разделены. Все пути относительны корню проекта, а seed и параметры модели зафиксированы в config.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import PipelineConfig
from src.data import load_data, build_validation_and_corpus, build_ranker_train, build_internal
from src.features import FINAL_FEATURE_NAMES, FIELD_FEATURE_NAMES, GEO_FEATURE_NAMES
from src.geo import build_transition_table
from src.pipeline import check_environment, reproduce
from src.retrieval import sparse_topk
from src.utils import normalize_text

ROOT = Path.cwd()
DATA = ROOT / "data"
CONFIG = PipelineConfig.from_root(ROOT, DATA, ROOT / "answer_v4.csv")
SEED = CONFIG.seed
check_environment(CONFIG)

### Загружаем данные

Сначала смотрим форму таблиц и несколько понятных колонок, не обучая модель.

In [ ]:
train_df, benchmark_queries_df, benchmark_items_df = load_data(DATA)
print("train:", train_df.shape)
print("benchmark queries:", benchmark_queries_df.shape)
print("benchmark items:", benchmark_items_df.shape)
display(train_df[["search_query", "item_title_raw", "item_id"]].head(3))

## 2. Что требуется и что видно в данных

Для каждого benchmark-запроса нужно вернуть 50 `item_id`. Recall@50 проверяет, не потеряли ли мы подходящие объявления. Ниже — только исследование данных: частотность запросов, заполненность фильтров и география корпуса. Эти расчёты не участвуют в модели.

In [ ]:
query_frequency = normalize_text(train_df.search_query).value_counts()
print(query_frequency.describe(percentiles=[.5, .9, .95, .99]))
print("Фильтры train:", normalize_text(train_df.search_infm_params_text).ne("").mean())
print("Фильтры benchmark:", normalize_text(benchmark_queries_df.search_infm_params_text).ne("").mean())

In [ ]:
geo = benchmark_items_df[["item_longitude", "item_latitude"]].apply(pd.to_numeric, errors="coerce").dropna()
plt.figure(figsize=(8, 5))
plt.hexbin(geo.item_longitude, geo.item_latitude, gridsize=100, bins="log", mincnt=1)
plt.xlabel("долгота")
plt.ylabel("широта")
plt.title("Расположение benchmark-объявлений")
plt.colorbar(label="log числа объявлений")
plt.show()

## 3. Validation без объединения независимых событий

В train нет `query_id`. Поэтому один пример — уникальная пара `(видимая сигнатура поиска, item_id)` с одним известным положительным объектом. Локальная метрика называется **single-positive offline Recall@50 proxy**.

Development, internal и корпус строятся детерминированно. Закрытый holdout после ранней проверки v1 для настройки больше не используется.

In [ ]:
split_dir = CONFIG.cache_dir / "v4" / "splits"
if (split_dir / "development.pkl").exists():
    development = pd.read_pickle(split_dir / "development.pkl")
    corpus = pd.read_pickle(split_dir / "corpus.pkl")
    ranker_train = pd.read_pickle(split_dir / "ranker_train.pkl")
else:
    development, _unused_holdout, corpus = build_validation_and_corpus(train_df, SEED)
    ranker_train = build_ranker_train(train_df, development, corpus, seed=SEED)
internal = build_internal(train_df, development, ranker_train, corpus, seed=SEED)
print({"train": len(ranker_train), "internal": len(internal), "development": len(development), "corpus": len(corpus)})

## 4. Первый baseline: TF-IDF по заголовку

Это первый понятный ориентир. Он ищет объявления по косинусному сходству запроса и заголовка, без нейросетей и метаданных.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, dtype=np.float32)
item_matrix = vectorizer.fit_transform(normalize_text(corpus.item_title_raw))
query_matrix = vectorizer.transform(development.search_query.fillna(""))
baseline_positions, baseline_scores = sparse_topk(query_matrix, item_matrix, top_k=50)
item_index = pd.Index(corpus.item_id.astype(str))
positive = item_index.get_indexer(development.item_id.astype(str))
baseline_hit50 = (baseline_positions == positive[:, None]).any(axis=1).mean()
print("TF-IDF single-positive Hit@50:", round(float(baseline_hit50), 4))

## 5. Почему одного источника мало

TF-IDF хорошо ловит точные слова, Giga — похожий смысл, а локальные ветки возвращают объявления, вытесненные в глобальной выдаче. Final v4 объединяет девять списков. Каждый источник сохраняет позиции и оценки; повторный запуск читает их из `cache/`.

In [ ]:
source_table = pd.DataFrame([
    ("Lexical", "TF-IDF по заголовку", "точные слова"),
    ("Giga", "весь корпус", "смысловое сходство"),
    ("Giga_local", "та же локация", "местные объявления"),
    ("Description", "описание", "подробности услуги"),
    ("Title_params", "заголовок и параметры", "характеристики"),
    ("Giga_local_deep", "та же локация глубже", "редкие местные items"),
    ("Description_local", "местные описания", "текст и локация"),
    ("Giga_geo50", "радиус 50 км", "соседние локации"),
    ("Geo_redirect", "исторические переходы", "межгородские случаи"),
], columns=["источник", "где ищет", "зачем"])
source_table

### Точный local и радиус

`Giga_local` ищет только в точной локации, `Giga_geo50` — около географического центра локации. Это мягкие источники кандидатов, а не жёсткие фильтры.

In [ ]:
from src.retrieval import restricted_dense, geo50_dense
print(restricted_dense.__doc__ or "Точный dense-поиск внутри группы location_id")
print(geo50_dense.__doc__ or "Dense-поиск среди объявлений в радиусе 50 км")

## 6. География: exact, радиус и поведение

По истории оцениваем, в каких локациях пользователи из каждой поисковой локации реально выбирали услуги. Это позволяет находить межгородские положительные примеры.

In [ ]:
local_items = train_df[[c for c in train_df if c == "item_id" or c.startswith("item_")]].drop_duplicates("item_id")
transitions, transition_summary = build_transition_table(train_df, local_items)
transition_summary[["observations", "top1_probability", "destinations", "entropy"]].describe()

## 7. Общий пул кандидатов

Кандидаты всех источников объединяются по позиции объявления. Получается несколько сотен уникальных items; это ещё не ответ, а вход для ранжирования. Final-модель должна выбрать 50.

In [ ]:
def union_positions(source_rows):
    return np.unique(np.concatenate([row[row >= 0] for row in source_rows]))

## 8. Признаки как несколько понятных семей

Не нужно запоминать 185 отдельных колонок. Это 131 повторяющийся базовый признак источников и метаданных, 11 признаков behavioral geo и 43 сравнения запроса с title, params и description.

In [ ]:
print("Всего признаков:", len(FINAL_FEATURE_NAMES))
print("Behavioral geo:", len(GEO_FEATURE_NAMES))
print("Field interactions:", len(FIELD_FEATURE_NAMES))
feature_families = pd.DataFrame({
    "семейство": ["источники и относительные оценки", "behavioral geo", "текстовые поля"],
    "число признаков": [131, len(GEO_FEATURE_NAMES), len(FIELD_FEATURE_NAMES)],
    "пример": ["rank, gap до top1, z-score", "transition probability", "редкое слово в title"],
})
feature_families

## 9. Один кандидат руками

Для объяснения одной строки не нужно пересчитывать весь pipeline. Берём сохранённый golden-пример и показываем несколько признаков одного реального кандидата.

In [ ]:
golden_path = ROOT / "tests" / "golden" / "v4_sample.npz"
with np.load(golden_path, allow_pickle=True) as golden:
    names = golden["feature_names"].tolist()
    row = golden["internal_0_features"][0]
    candidate_id = golden["internal_0_candidate_ids"][0]

shown = ["lexical_rank", "giga_rank", "same_location", "matched_query_idf_fraction", "transition_probability"]
pd.Series({name: row[names.index(name)] for name in shown}, name=str(candidate_id)).to_frame("значение")

## 10. Как обучается YetiRank

Одна группа соответствует одному запросу: внутри есть выбранное объявление и кандидаты, которые реально нашла retrieval-система. YetiRank учится поднимать выбранный item относительно остальных. Параметры frozen-модели показаны ниже.

In [ ]:
CONFIG.catboost_params

## 11. Контрольные результаты

Frozen E4 должен воспроизводить internal Hit@50 = 0.8878, development Hit@50 = 0.8940 и development coverage = 0.9642. Полная проверка дорогая, поэтому включается осознанно.

In [ ]:
RUN_LOCAL_EVALUATION = False
if RUN_LOCAL_EVALUATION:
    local_report = reproduce(CONFIG, retrain=True)
    display(local_report["local_metrics"])
else:
    print("Для полного пересчёта: RUN_LOCAL_EVALUATION = True")

## 12. Final inference и answer_v4.csv

Следующий вызов запускает реальный pipeline. С готовым кэшем он читает сохранённые дорогие стадии, создаёт `answer_v4.csv` и строго проверяет его формат. Полученный файл дал официальный Recall@50 **0.844391**; benchmark не использовался для выбора признаков или параметров.

```python
run_report = reproduce(CONFIG, retrain=False)
run_report
```

In [ ]:
RUN_FINAL_INFERENCE = False
if RUN_FINAL_INFERENCE:
    final_report = reproduce(CONFIG, retrain=False)
    display(final_report)
else:
    print("Для создания answer_v4.csv: RUN_FINAL_INFERENCE = True")